# Build balanced and unbalanced macro panels

This notebook rebuilds the two model-ready feature panels from
`deterministic_data2_policy_extended_COPY.xlsx`.

Definitions:
- Surprise = Actual − Bloomberg Median consensus.
- Standardization uses the **prior 12 releases only** and sample standard deviation.
- Unbalanced panel retains structural history and assigns era-specific feature sets.
- Balanced panel keeps only months where all eight macro surprise features and both policy-rate features are observed.
- The source workbook is not overwritten.

In [ ]:
from artifact_tool import Blob, SpreadsheetFile
from pathlib import Path
from datetime import datetime, timedelta
import calendar, csv, math, statistics

BASE = Path("/mnt/data")
SOURCE = BASE / "deterministic_data2_policy_extended_COPY.xlsx"
wb = SpreadsheetFile.import_xlsx(Blob.load(str(SOURCE)))

In [ ]:
sheet_cfg = {
    "nfp": ("nfp_release_raw", "2000-01-07"),
    "retail_sales": ("retail_sales_release_raw", "2001-06-13"),
    "industrial_production": ("industrial_production_release_r", "2000-01-14"),
    "ism_manufacturing": ("ism_manufacturing_release", "2000-01-03"),
    "cpi": ("cpi_release_raw", "2000-01-14"),
    "core_cpi": ("core_cpi_release_raw", "2000-01-14"),
    "ppi": ("ppi_release_raw", "2014-02-19"),
    "ahe": ("ahe_release_raw", "2010-02-05"),
}
feature_order = list(sheet_cfg)

def isnum(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v)

def excel_date(v):
    if isinstance(v, (int, float)):
        return datetime(1899, 12, 30) + timedelta(days=float(v))
    if isinstance(v, str) and v and not v.startswith("#"):
        try:
            return datetime.fromisoformat(v.replace("Z",""))
        except Exception:
            return None
    return None

def month_end_dt(ym):
    y,m = map(int, ym.split("-"))
    return datetime(y,m,calendar.monthrange(y,m)[1])

In [ ]:
def standardized_surprises(sheet_name, first_date):
    vals = wb.worksheets.get_item(sheet_name).get_range("A1:G400").values
    obs = []
    for row_num, r in enumerate(vals[1:], start=2):
        if len(r) < 3 or not isnum(r[1]) or not isnum(r[2]):
            continue
        dt = excel_date(r[0])
        # The first Bloomberg array-formula date is cached as #NAME? outside Bloomberg.
        if dt is None and row_num == 2:
            dt = datetime.fromisoformat(first_date)
        if dt is None:
            continue
        obs.append((dt, float(r[1]) - float(r[2])))
    obs.sort(key=lambda x: x[0])

    result = []
    for i in range(12, len(obs)):
        previous = [x[1] for x in obs[i-12:i]]
        sd = statistics.stdev(previous)
        z = None if sd == 0 else (obs[i][1] - statistics.mean(previous)) / sd
        result.append((obs[i][0], z))
    return result

computed = {
    f: standardized_surprises(sheet, first_date)
    for f,(sheet,first_date) in sheet_cfg.items()
}

In [ ]:
# Collapse each release series to one value per calendar month.
macro_monthly = {}
for feat in feature_order:
    monthly = {}
    for dt,z in computed[feat]:
        if z is None:
            continue
        ym = dt.strftime("%Y-%m")
        if ym not in monthly or dt > monthly[ym][0]:
            monthly[ym] = (dt, float(z))
    macro_monthly[feat] = monthly

def policy_map(sheet_name):
    vals = wb.worksheets.get_item(sheet_name).get_range("A1:B400").values
    out = {}
    for r in vals[1:]:
        if len(r) < 2 or not isnum(r[1]):
            continue
        dt = excel_date(r[0])
        if dt is not None:
            out[dt.strftime("%Y-%m")] = float(r[1])
    return out

policy3 = policy_map("policy_3m_extended")
policy6 = policy_map("policy_6m_extended")

In [ ]:
era_defs = [
    ("E1_core5", "2001-01", "2002-05",
     ["nfp","industrial_production","ism_manufacturing","cpi","core_cpi"]),
    ("E2_core6", "2002-06", "2011-02",
     ["nfp","retail_sales","industrial_production","ism_manufacturing","cpi","core_cpi"]),
    ("E3_plus_ahe", "2011-03", "2015-01",
     ["nfp","retail_sales","industrial_production","ism_manufacturing","cpi","core_cpi","ahe"]),
    ("E4_full8", "2015-02", "2026-09",
     ["nfp","retail_sales","industrial_production","ism_manufacturing","cpi","core_cpi","ppi","ahe"]),
]

def era_for_month(ym):
    for eid,start,end,features in era_defs:
        if start <= ym <= end:
            return eid, features
    return None, []

def month_range(start, end):
    y,m = map(int,start.split("-"))
    ey,em = map(int,end.split("-"))
    out=[]
    while (y,m) <= (ey,em):
        out.append(f"{y:04d}-{m:02d}")
        y,m = (y+1,1) if m==12 else (y,m+1)
    return out

months = month_range(
    min(min(x) for x in macro_monthly.values()),
    min(max(policy3), max(policy6))
)

In [ ]:
unbalanced = []
for ym in months:
    row = {"date": month_end_dt(ym), "month": ym}
    active = []
    for feat in feature_order:
        val = macro_monthly[feat].get(ym)
        row[feat+"_z"] = val[1] if val else None
        if val:
            active.append(feat)

    row["policy_3m"] = policy3.get(ym)
    row["policy_6m"] = policy6.get(ym)
    row["policy_slope_6m_minus_3m"] = (
        row["policy_6m"] - row["policy_3m"]
        if row["policy_3m"] is not None and row["policy_6m"] is not None else None
    )

    era_id, expected = era_for_month(ym)
    row["era_id"] = era_id
    row["n_macro_available"] = len(active)
    row["row_complete_for_era"] = all(row[f+"_z"] is not None for f in expected)
    row["expected_macro_set"] = ", ".join(expected)
    row["active_macro_set"] = ", ".join(active)
    unbalanced.append(row)

balanced = [
    r.copy() for r in unbalanced
    if all(r[f+"_z"] is not None for f in feature_order)
    and r["policy_3m"] is not None
    and r["policy_6m"] is not None
]

print("Unbalanced:", len(unbalanced), unbalanced[0]["month"], "to", unbalanced[-1]["month"])
print("Balanced:", len(balanced), balanced[0]["month"], "to", balanced[-1]["month"])

In [ ]:
# Save model-ready CSVs.
unbal_fields = [
    "date","month","era_id","row_complete_for_era","n_macro_available",
    "nfp_z","retail_sales_z","industrial_production_z","ism_manufacturing_z",
    "cpi_z","core_cpi_z","ppi_z","ahe_z",
    "policy_3m","policy_6m","policy_slope_6m_minus_3m",
    "expected_macro_set","active_macro_set",
]
bal_fields = [
    "date","month",
    "nfp_z","retail_sales_z","industrial_production_z","ism_manufacturing_z",
    "cpi_z","core_cpi_z","ppi_z","ahe_z",
    "policy_3m","policy_6m","policy_slope_6m_minus_3m",
]

for path, rows, fields in [
    (BASE/"model_unbalanced_panel.csv", unbalanced, unbal_fields),
    (BASE/"model_balanced_panel.csv", balanced, bal_fields),
]:
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            rr = {k:r.get(k) for k in fields}
            rr["date"] = rr["date"].date().isoformat()
            w.writerow(rr)

print("Saved CSVs.")

## Backtest usage

For the **unbalanced panel**, do not send NaNs directly into vanilla `sklearn.Ridge`.
Use the predeclared era-specific feature sets and train each historical prediction only on rows that are complete for that era.

For the **balanced panel**, use the same eight macro features and policy features for every observation.

When comparing the approaches, report:
1. each specification on its full practical sample, and
2. both specifications on the same post-2015 dates.